In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from model import GraphSAGE

c:\Users\tusha\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
checkpoint = torch.load('models/task_a_model.pt', weights_only=False)
data = checkpoint['data']
student_id_map = checkpoint['student_id_map']
course_id_map = checkpoint['course_id_map']
concept_id_map = checkpoint['concept_id_map']

model = GraphSAGE(hidden_dim=256)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

enrollments = pd.read_csv('datasets/enrollments.csv')
courses = pd.read_csv('datasets/courses.csv')
chatbot = pd.read_csv('datasets/chatbot_signals.csv')
assessments = pd.read_csv('datasets/assessment_scores.csv')
course_concepts_df = pd.read_csv('datasets/course_concepts.csv')
concepts = pd.read_csv('datasets/concepts.csv')

In [4]:
course_name_map = courses.set_index('course_id')['course_name'].to_dict()
concept_name_map = concepts.set_index('concept_id')['concept_name'].to_dict()

failing = enrollments[enrollments['grade'] < 1.5]

course_stats = enrollments.groupby('course_id').agg(
    total_enrolled = ('student_id', 'count'),
    avg_grade = ('grade', 'mean'),
    fail_count = ('passed', lambda x: (x==0).sum()),
    fail_rate = ('passed', lambda x: (x==0).mean()),
).reset_index()

course_stats['course_name'] = course_stats['course_id'].map(course_name_map)
course_stats = course_stats.sort_values('fail_rate', ascending = False)

print("Top 10 courses by failure rate: ")
print(course_stats[['course_name', 'total_enrolled', 'fail_count', 'fail_rate', 'avg_grade']].head(10).to_string(index = False))

Top 10 courses by failure rate: 
                 course_name  total_enrolled  fail_count  fail_rate  avg_grade
         Full Stack Capstone             541          72   0.133087   2.012957
Product Engineering Capstone             501          61   0.121756   2.024291
              Cloud Capstone             484          57   0.117769   2.031942
   Data Engineering Capstone             476          55   0.115546   1.995861
   Research Methods Capstone             528          61   0.115530   2.015928
            Systems Capstone             495          54   0.109091   2.046667
               Deep Learning             746          81   0.108579   2.093794
  Cybersecurity Fundamentals             760          81   0.106579   2.079789
      Cybersecurity Capstone             472          50   0.105932   2.038157
        Software Engineering             768          81   0.105469   2.077318


In [5]:
failing_students = failing['student_id'].unique()

failing_chatbot = chatbot[chatbot['student_id'].isin(failing_students)]

concept_to_course = course_concepts_df.groupby('concept_id')['course_id'].apply(list).to_dict()

results = []
for course_id in course_stats['course_id']:
    course_concepts = course_concepts_df[course_concepts_df['course_id'] == course_id]['concept_id'].tolist()

    course_failing = failing[failing['course_id'] == course_id]['student_id'].tolist()

    relevant = chatbot[
        (chatbot['student_id'].isin(course_failing)) &
        (chatbot['concept_id'].isin(course_concepts))
    ]

    if len(relevant) > 0:
        concept_confusion = relevant.groupby('concept_id')['confusion_score'].mean().sort_values(ascending=False)
        
        for concept_id, confusion in concept_confusion.head(3).items():
            results.append({
                'course': course_name_map[course_id],
                'concept': concept_name_map[concept_id], 
                'avg_confusion': round(confusion, 3)
            })

results_df = pd.DataFrame(results)
print("Top 3 most confusing concepts per struggling course:")
print(results_df.to_string(index=False))

Top 3 most confusing concepts per struggling course:
                      course                        concept  avg_confusion
         Full Stack Capstone               Model Deployment          0.470
         Full Stack Capstone            APIs and Interfaces          0.469
         Full Stack Capstone                  Microservices          0.435
Product Engineering Capstone               Model Deployment          0.482
Product Engineering Capstone              Usability Testing          0.279
Product Engineering Capstone                    Prototyping          0.264
              Cloud Capstone                 Cloud Services          0.625
              Cloud Capstone                Fault Tolerance          0.545
              Cloud Capstone               Containerization          0.544
   Data Engineering Capstone                 Data Pipelines          0.621
   Data Engineering Capstone                 Cloud Services          0.592
   Data Engineering Capstone               Data

In [6]:
model.eval()
student_indices = torch.tensor([student_id_map[s] for s in enrollments['student_id']], dtype = torch.long)
course_indices = torch.tensor([course_id_map[c] for c in enrollments['course_id']], dtype = torch.long)

with torch.no_grad():
    out = model(data)
    student_embs = out[0][student_indices]
    course_embs = out[1][course_indices]
    pairs = torch.cat([student_embs, course_embs, student_embs - course_embs, student_embs * course_embs], dim = 1)
    baseline_preds = model.task_a_head(pairs).argmax(dim = 1)

print(f"Baseline total High risk: {(baseline_preds == 2).sum().item()}")

Baseline total High risk: 4953


In [7]:
confusion_lookup = chatbot.set_index(['student_id','concept_id'])['confusion_score'].to_dict()
print(f"Total student-concept pairs in lookup: {len(confusion_lookup)}")

Total student-concept pairs in lookup: 35006


In [17]:
def simulate_concept_tutoring(course_id, reduction = 0.5):
    course_name = course_name_map[course_id]
    course_concepts = course_concepts_df[course_concepts_df['course_id'] == course_id]['concept_id'].tolist()

    course_enrollments = enrollments[enrollments['course_id'] == course_id]
    course_students = course_enrollments['student_id'].tolist()

    course_mask = (enrollments['course_id'] == course_id).values
    baseline_count = (baseline_preds[course_mask] == 2).sum().item()

    concept_results = []

    for concept_id in course_concepts:
        affected_students = []
        for student_id in course_students:
            conf = confusion_lookup.get((student_id, concept_id), None)
            if conf is not None:
                affected_students.append(student_id_map[student_id])

        if len(affected_students) ==0:
            continue

        original_student_x = data['student'].x.clone()

        for s_idx in affected_students:
            data['student'].x[s_idx, 3] = data['student'].x[s_idx, 3] * reduction

        with torch.no_grad():
            out = model(data)
            pairs = torch.cat([out[0][student_indices], out[1][course_indices],
                               out[0][student_indices] - out[1][course_indices],
                               out[0][student_indices] * out[1][course_indices]], dim=1)
            new_preds = model.task_a_head(pairs).argmax(dim = 1)

        new_count = (new_preds[course_mask] ==2 ).sum().item()
        helped = baseline_count - new_count

        concept_results.append({
            'concept': concept_name_map[concept_id],
            'students_affected': len(affected_students),
            'students_helped': helped
        })

        data['student'].x = original_student_x

    concept_results.sort(key = lambda x: x['students_helped'], reverse=True)

    return course_name, baseline_count, concept_results

In [18]:
print("=== Course Improvement Recommendations ===\n")

for course_id in course_stats['course_id'].head(10):
    course_name, baseline, concept_results = simulate_concept_tutoring(course_id)
    
    print(f"{course_name} (Baseline High-risk: {baseline})")
    
    # Show top 3 recommendations
    for i, result in enumerate(concept_results[:3]):
        print(f"  {i+1}. Add tutoring for {result['concept']}")
        print(f"     Students affected: {result['students_affected']}")
        print(f"     Students helped: {result['students_helped']}")

=== Course Improvement Recommendations ===

Full Stack Capstone (Baseline High-risk: 100)
  1. Add tutoring for Testing and Validation
     Students affected: 158
     Students helped: 13
  2. Add tutoring for Microservices
     Students affected: 161
     Students helped: 12
  3. Add tutoring for APIs and Interfaces
     Students affected: 156
     Students helped: 11
Product Engineering Capstone (Baseline High-risk: 86)
  1. Add tutoring for Design Systems
     Students affected: 165
     Students helped: 16
  2. Add tutoring for Usability Testing
     Students affected: 165
     Students helped: 10
  3. Add tutoring for Prototyping
     Students affected: 134
     Students helped: 8
Cloud Capstone (Baseline High-risk: 73)
  1. Add tutoring for Fault Tolerance
     Students affected: 134
     Students helped: 16
  2. Add tutoring for Containerization
     Students affected: 141
     Students helped: 12
  3. Add tutoring for Cloud Services
     Students affected: 149
     Students hel